In [2]:
# Import libs
import pandas as pd
from sklearn.cluster import KMeans
import folium

# Create fake stops data (lat, lon)
data = {
    'stop_id': [1, 2, 3, 4, 5, 6],
    'stop_name': ['Stop A', 'Stop B', 'Stop C', 'Stop D', 'Stop E', 'Stop F'],
    'latitude': [40.7128, 40.7138, 40.7148, 40.7158, 40.7168, 40.7178],
    'longitude': [-74.0060, -74.0050, -74.0040, -74.0030, -74.0020, -74.0010]
}
df = pd.DataFrame(data)

# Run KMeans clustering with 2 clusters
kmeans = KMeans(n_clusters=2, random_state=42)
df['cluster'] = kmeans.fit_predict(df[['latitude', 'longitude']])

# Create a folium map centered on average location
map_center = [df['latitude'].mean(), df['longitude'].mean()]
mymap = folium.Map(location=map_center, zoom_start=15)

# Add stops as markers colored by cluster
colors = ['red', 'blue', 'green', 'purple', 'orange']
for idx, row in df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=7,
        color=colors[row['cluster']],
        fill=True,
        fill_color=colors[row['cluster']],
        popup=f"{row['stop_name']} (Cluster {row['cluster']})"
    ).add_to(mymap)

# Display map
mymap


In [3]:
import numpy as np

# Function to get nearest neighbor route for points in a cluster
def nearest_neighbor_route(df_cluster):
    coords = df_cluster[['latitude', 'longitude']].to_numpy()
    n = len(coords)
    visited = [False]*n
    route = [0]  # start from first point
    visited[0] = True

    for _ in range(n-1):
        last = route[-1]
        dists = np.linalg.norm(coords - coords[last], axis=1)
        dists = [d if not visited[i] else np.inf for i, d in enumerate(dists)]
        next_idx = np.argmin(dists)
        route.append(next_idx)
        visited[next_idx] = True

    return route

# Show routes per cluster
for cluster_id in df['cluster'].unique():
    cluster_points = df[df['cluster'] == cluster_id].reset_index(drop=True)
    route = nearest_neighbor_route(cluster_points)
    print(f"Cluster {cluster_id} route stop order:")
    for idx in route:
        print(f"  {cluster_points.loc[idx, 'stop_name']}")


Cluster 0 route stop order:
  Stop A
  Stop B
  Stop C
Cluster 1 route stop order:
  Stop D
  Stop E
  Stop F
